# Reciprocal accuracy from testbench CSVs

Reads the CSV written by each common-pipeline testbench and computes the two accuracy blocks of the results table:

| Block | Reference | Meaning |
|---|---|---|
| **Hardware** | quantized golden value | `HW_Q` vs `Golden_Q` |
| **Mathematical** | exact real-valued 1/x | `HW` vs `1/x` |

Both blocks use the same 7 metrics, in the sheet's column order:
`Max Abs Error (LSB)`, `MAE (LSB)`, `RMSE (LSB)`, `Max Abs Error (Real)`, `MAE (Real)`, `Max Relative Error (%)`, `Mean Relative Error (%)`.

Everything is recomputed from the hex columns (`x_hex`, `golden_hex`, `hw_hex`), so nothing depends on the rounding of the printed real columns.
Run the cells top to bottom.

## 1. Settings

In [1]:
# One entry per algorithm: label (as in the sheet) -> CSV path.
# Paths are relative to this notebook; edit them to point at your files.
CSV_FILES = {
    "Direct LUT":          "C:\\Users\\knith\\OneDrive\\Desktop\\direct_lut_results.csv",
    "LUT + Interpolation": "C:\\Users\\knith\\OneDrive\\Desktop\\interp_lut_results.csv",
    "Newton-Raphson":      "C:\\Users\\knith\\OneDrive\\Desktop\\q8_24_nr_results.csv",
    "Goldschmidt":         "C:\\Users\\knith\\OneDrive\\Desktop\\q8_24_gs_results.csv",
    "Hybrid":              "C:\\Users\\knith\\OneDrive\\Desktop\\hybrid9_1nr_results.csv",
}

FRAC_BITS = 24      # 24 for Q8.24, 16 for Q16.16

SAVE_SUMMARY_TO = "accuracy_summary.csv"   # set to None to skip saving

## 2. Load and compute

In [2]:
import os
import numpy as np
import pandas as pd

SCALE = float(1 << FRAC_BITS)          # 2^F  (1 LSB = 1/SCALE)
TWO_F = float(1 << (2 * FRAC_BITS))    # 2^(2F): exact reciprocal of x_int, in LSB, is TWO_F / x_int

METRICS = ["Max Abs Error (LSB)", "MAE (LSB)", "RMSE (LSB)",
           "Max Abs Error (Real)", "MAE (Real)",
           "Max Relative Error (%)", "Mean Relative Error (%)"]


def hex_col(series):
    """Hex strings -> int64 array. Rows that are not valid hex (e.g. 'xxxxxxxx') come back as NaN mask."""
    ok = series.str.fullmatch(r"[0-9a-fA-F]+").fillna(False).to_numpy()
    vals = np.zeros(len(series), dtype=np.int64)
    vals[ok] = [int(s, 16) for s in series[ok]]
    return vals, ok


def summarize(abs_err_lsb, rel_err):
    mae = abs_err_lsb.mean()
    return [
        abs_err_lsb.max(),                       # Max Abs Error (LSB)
        mae,                                     # MAE (LSB)
        np.sqrt((abs_err_lsb ** 2).mean()),      # RMSE (LSB)
        abs_err_lsb.max() / SCALE,               # Max Abs Error (Real)
        mae / SCALE,                             # MAE (Real)
        100.0 * rel_err.max(),                   # Max Relative Error (%)
        100.0 * rel_err.mean(),                  # Mean Relative Error (%)
    ]


def analyse(path):
    df = pd.read_csv(path, dtype=str)
    x, ok_x = hex_col(df["x_hex"])
    g, ok_g = hex_col(df["golden_hex"])
    h, ok_h = hex_col(df["hw_hex"])
    ok = ok_x & ok_g & ok_h
    skipped = int((~ok).sum())
    x, g, h = x[ok], g[ok], h[ok]

    # Hardware vs quantized golden: integer LSB error
    hw_err = np.abs(h - g).astype(np.float64)
    hw_rel = np.divide(hw_err, g, out=np.zeros_like(hw_err), where=g != 0)

    # Hardware vs exact 1/x: error in (fractional) LSB
    nz = x != 0
    ideal_lsb = TWO_F / x[nz]
    m_err = np.abs(h[nz] - ideal_lsb)
    m_rel = m_err / ideal_lsb

    return summarize(hw_err, hw_rel), summarize(m_err, m_rel), len(h), skipped, int((~nz).sum())


rows = []
for label, path in CSV_FILES.items():
    if not os.path.exists(path):
        print(f"[skip] {label}: {path} not found")
        continue
    hw, mt, n, skipped, zero_x = analyse(path)
    if skipped:
        print(f"[warn] {label}: {skipped} row(s) with non-hex output skipped (X/Z)")
    if zero_x:
        print(f"[warn] {label}: {zero_x} row(s) with x = 0 excluded from the mathematical block")
    rows.append({"Algorithm": label, "Block": "Hardware (vs golden)", "Rows": n, **dict(zip(METRICS, hw))})
    rows.append({"Algorithm": label, "Block": "Mathematical (vs 1/x)", "Rows": n, **dict(zip(METRICS, mt))})

results = pd.DataFrame(rows)
print(f"computed {len(results) // 2} algorithm(s)")

computed 5 algorithm(s)


## 3. Results

In [3]:
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", None)

fmt = {"Max Abs Error (LSB)": "{:.4f}", "MAE (LSB)": "{:.6f}", "RMSE (LSB)": "{:.6f}",
       "Max Abs Error (Real)": "{:.6e}", "MAE (Real)": "{:.6e}",
       "Max Relative Error (%)": "{:.6e}", "Mean Relative Error (%)": "{:.6e}"}

results.style.format(fmt).hide(axis="index")

Algorithm,Block,Rows,Max Abs Error (LSB),MAE (LSB),RMSE (LSB),Max Abs Error (Real),MAE (Real),Max Relative Error (%),Mean Relative Error (%)
Direct LUT,Hardware (vs golden),1000001,32716.0000,3277.242250,5440.197673,1.950026e-03,1.953389e-04,9.759644e-02,1.965007e-02
Direct LUT,Mathematical (vs 1/x),1000001,32716.0702,3277.241972,5440.197648,1.950030e-03,1.953388e-04,9.759665e-02,1.965007e-02
LUT + Interpolation,Hardware (vs golden),1000001,32.0000,2.315201,5.029770,1.907349e-06,1.379967e-07,9.565506e-05,1.126783e-05
LUT + Interpolation,Mathematical (vs 1/x),1000001,31.9603,2.360406,5.021749,1.904981e-06,1.406912e-07,9.552754e-05,1.177460e-05
Newton-Raphson,Hardware (vs golden),999977,2.0000,0.202515,0.450023,1.192093e-07,1.207081e-08,1.490112e-05,1.523345e-06
Newton-Raphson,Mathematical (vs 1/x),999977,1.5653,0.314278,0.386642,9.329810e-08,1.873240e-08,1.025044e-05,2.631672e-06
Goldschmidt,Hardware (vs golden),999971,3.0000,0.361510,0.615260,1.788139e-07,2.154770e-08,1.966612e-05,2.919497e-06
Goldschmidt,Mathematical (vs 1/x),999971,2.7496,0.431763,0.544494,1.638914e-07,2.573510e-08,1.933359e-05,3.592342e-06
Hybrid,Hardware (vs golden),999953,137.0000,26.844360,38.597141,8.165836e-06,1.600049e-06,4.093354e-04,1.609460e-04
Hybrid,Mathematical (vs 1/x),999953,136.7311,26.844656,38.596157,8.149811e-06,1.600066e-06,4.083780e-04,1.609487e-04


### Hardware block only  /  Mathematical block only

In [4]:
hw_table = results[results["Block"].str.startswith("Hardware")].drop(columns="Block").set_index("Algorithm")
math_table = results[results["Block"].str.startswith("Mathematical")].drop(columns="Block").set_index("Algorithm")

print("HARDWARE  (HW_Q vs quantized golden)")
display(hw_table.style.format(fmt))
print("MATHEMATICAL  (HW vs exact 1/x)")
display(math_table.style.format(fmt))

HARDWARE  (HW_Q vs quantized golden)


,Rows,Max Abs Error (LSB),MAE (LSB),RMSE (LSB),Max Abs Error (Real),MAE (Real),Max Relative Error (%),Mean Relative Error (%)
Algorithm,,,,,,,,
Direct LUT,1000001,32716.0000,3277.242250,5440.197673,1.950026e-03,1.953389e-04,9.759644e-02,1.965007e-02
LUT + Interpolation,1000001,32.0000,2.315201,5.029770,1.907349e-06,1.379967e-07,9.565506e-05,1.126783e-05
Newton-Raphson,999977,2.0000,0.202515,0.450023,1.192093e-07,1.207081e-08,1.490112e-05,1.523345e-06
Goldschmidt,999971,3.0000,0.361510,0.615260,1.788139e-07,2.154770e-08,1.966612e-05,2.919497e-06
Hybrid,999953,137.0000,26.844360,38.597141,8.165836e-06,1.600049e-06,4.093354e-04,1.609460e-04


MATHEMATICAL  (HW vs exact 1/x)


,Rows,Max Abs Error (LSB),MAE (LSB),RMSE (LSB),Max Abs Error (Real),MAE (Real),Max Relative Error (%),Mean Relative Error (%)
Algorithm,,,,,,,,
Direct LUT,1000001,32716.0702,3277.241972,5440.197648,1.950030e-03,1.953388e-04,9.759665e-02,1.965007e-02
LUT + Interpolation,1000001,31.9603,2.360406,5.021749,1.904981e-06,1.406912e-07,9.552754e-05,1.177460e-05
Newton-Raphson,999977,1.5653,0.314278,0.386642,9.329810e-08,1.873240e-08,1.025044e-05,2.631672e-06
Goldschmidt,999971,2.7496,0.431763,0.544494,1.638914e-07,2.573510e-08,1.933359e-05,3.592342e-06
Hybrid,999953,136.7311,26.844656,38.596157,8.149811e-06,1.600066e-06,4.083780e-04,1.609487e-04


## 4. Paste-ready rows

Tab-separated, in the sheet's column order. Copy a line and paste it into the first cell of the matching block
(Hardware block starts at `Max Abs Error (LSB)`, Mathematical block likewise).

In [5]:
def line(vals):
    return "\t".join(fmt[m].format(v) for m, v in zip(METRICS, vals))

for label in results["Algorithm"].unique():
    sub = results[results["Algorithm"] == label]
    hw_vals = sub[sub["Block"].str.startswith("Hardware")][METRICS].iloc[0].tolist()
    mt_vals = sub[sub["Block"].str.startswith("Mathematical")][METRICS].iloc[0].tolist()
    print(f"{label}")
    print("  Hardware     :", line(hw_vals))
    print("  Mathematical :", line(mt_vals))

Direct LUT
  Hardware     : 32716.0000	3277.242250	5440.197673	1.950026e-03	1.953389e-04	9.759644e-02	1.965007e-02
  Mathematical : 32716.0702	3277.241972	5440.197648	1.950030e-03	1.953388e-04	9.759665e-02	1.965007e-02
LUT + Interpolation
  Hardware     : 32.0000	2.315201	5.029770	1.907349e-06	1.379967e-07	9.565506e-05	1.126783e-05
  Mathematical : 31.9603	2.360406	5.021749	1.904981e-06	1.406912e-07	9.552754e-05	1.177460e-05
Newton-Raphson
  Hardware     : 2.0000	0.202515	0.450023	1.192093e-07	1.207081e-08	1.490112e-05	1.523345e-06
  Mathematical : 1.5653	0.314278	0.386642	9.329810e-08	1.873240e-08	1.025044e-05	2.631672e-06
Goldschmidt
  Hardware     : 3.0000	0.361510	0.615260	1.788139e-07	2.154770e-08	1.966612e-05	2.919497e-06
  Mathematical : 2.7496	0.431763	0.544494	1.638914e-07	2.573510e-08	1.933359e-05	3.592342e-06
Hybrid
  Hardware     : 137.0000	26.844360	38.597141	8.165836e-06	1.600049e-06	4.093354e-04	1.609460e-04
  Mathematical : 136.7311	26.844656	38.596157	8.149811e-06	1.60

## 5. Save

In [6]:
if SAVE_SUMMARY_TO:
    results.to_csv(SAVE_SUMMARY_TO, index=False)
    print("saved", SAVE_SUMMARY_TO)

saved accuracy_summary.csv
